Danielle Fischer
DSAN5550

# Preliminary Results Analysis

## Dataset

The final dataset consists of county level observations from 2000-2022 across two regions: the corn belt and the great plains. The dataset integrates three sources:

- **USDA yield data** (target variable: yield in bushels per acre)
- **PRISM climate data**, aggregated over the growing season (May–September)
  - Mean temperature
  - Total precipitation
- **U.S. Drought Monitor data**
  - Frequency of severe drought (D2+)
  - Intensity of severe drought

After merging and filtering, the dataset contains:
- ~21,000 county-year observations
- ~1,100 counties
- complete coverage across all variables

The goal is to model crop yield as a function of climate and drought conditions, and evaluate how well these relationships transfer across regions.

Yield distributions differ substantially across regions. The Great Plains shows lower average yields and greater variability, including extreme low-yield observations. While these differences in scale may contribute to transfer challenges, they do not fully explain the observed results. Models trained in the Corn Belt produce negative R2 when applied to the Great Plains, indicating that the issue is not just a shift in avg yield, but a mismatch in the underlying relationships between climate variables and yield. The higher variability in the Great Plains suggests a more heterogeneous system, which may explain why models trained there generalize slightly better to the more stable Corn Belt.

In [7]:
import pandas as pd

df = pd.read_csv("../data/clean/final_model_data.csv")

cb = df[df["state"].isin([
    "ILLINOIS","INDIANA","IOWA","MINNESOTA","MISSOURI","OHIO","WISCONSIN"
])]
gp = df[df["state"].isin([
    "KANSAS","NEBRASKA","NORTH DAKOTA","OKLAHOMA","SOUTH DAKOTA","TEXAS"
])]

cb["yield_bu_acre"].describe(), gp["yield_bu_acre"].describe()

(count    13084.000000
 mean       154.989300
 std         33.000169
 min         19.000000
 25%        135.400000
 50%        157.850000
 75%        178.100000
 max        246.700000
 Name: yield_bu_acre, dtype: float64,
 count    8318.000000
 mean      121.425415
 std        44.896519
 min         0.000000
 25%        88.800000
 50%       119.450000
 75%       155.800000
 max       240.700000
 Name: yield_bu_acre, dtype: float64)

In [9]:
df = pd.read_csv("../data/clean/final_model_data.csv")

print("Dataset Summary")
print("----------------")
print(f"Number of rows: {df.shape[0]:,}")
print(f"Number of columns: {df.shape[1]}")
print(f"Number of states: {df['state'].nunique()}")
print(f"Year range: {df['year'].min()}–{df['year'].max()}")

Dataset Summary
----------------
Number of rows: 21,402
Number of columns: 11
Number of states: 13
Year range: 2000–2022


## Modeling Approach

Three modeling approaches were used:

### 1. Linear Regression
A baseline model to assess whether crop yield can be explained using simple linear relationships between climate and drought variables

### 2. Random Forest
An ensemble tree-based model capable of capturing nonlinear relationships and interactions between features. This serves as the primary model for interpretation

### 3. Gradient Boosting
A more flexible boosting-based model that sequentially improves predictions and can capture complex nonlinear patterns

### Evaluation Strategy

Models were evaluated using:

- **5-fold cross-validation** within regions
- **Transfer evaluations**, where models trained in one region were tested in another:
    - Corn Belt --> Great Plains
    - Great Plains --> Corn Belt

Performance was measured using:
- RMSE
- MAE
- R2

## Model Results

## A. Linear Regression Results

Linear regression performed poorly across all settings.

- Within the Corn Belt, R2 values were low (~0.05-0.11), indicating limited explanatory power
- Transfer performance was negative in both directions, with R2 values below zero

This suggests that crop yield is not well explained by simple linear relationships, and that regional differences cannot be captured with a shared linear model


## B. Random Forest Results

Random foerst substantially improved performance relative to linear regression

- Within-region performance was strong:
    - Corn Belt: R2 about 0.31 (with drought)
    - Great Plains: R2 about 0.36 (with drought)
- Including drought variables consistently improved predictions

Transfer results revealed strong asymmetry:
- Corn Belt to Great Plains had a negative R2 (model fails to generalize)
- Great Plains to Corn Belt had a slightly positive R2 (~0.05-0.09)

This indicates that relationships learned in the Great Plains are somewhat transferable, but the reverse is not true.

## C. Gradient Boosting

Gradient boosting achieved similar within-region performance to random forest, confirming the importance of nonlinear modeling.

- Corn Belt R2 was about 0.29
- Performance improves with drought features

However, transfer performance remained poor:
- Corn Belt to Great Plains had a strongly negative R2
- Great Plains to Corn Belt had a slightly positive R2

Compared to random forest, gradient boosting showed slightly worse transfer performance, suggesting it may overfit to regional patterns

In [25]:
metrics = pd.concat([
    pd.read_csv("../outputs/linear_regression_metrics.csv"),
    pd.read_csv("../outputs/random_forest_metrics.csv"),
    pd.read_csv("../outputs/gradient_boost_metrics.csv"),
])

**Corn Belt CV**

In [27]:
cb_cv = metrics[metrics["evaluation_set"] == "corn_belt_cv"]

cb_cv.pivot(
    index="model",
    columns="feature_set",
    values="r2"
).round(3)

feature_set,climate_only,climate_plus_drought
model,,
gradient_boost,0.219,0.294
linear_regression,0.045,0.113
random_forest,0.209,0.307


**Great Plains CV**

In [26]:
gp_cv = metrics[metrics["evaluation_set"] == "great_plains_cv"]

gp_cv.pivot(
    index="model",
    columns="feature_set",
    values="r2"
).round(3)

feature_set,climate_only,climate_plus_drought
model,,
random_forest,0.306,0.356


**Transfer results**

In [22]:
transfer_table.style.background_gradient(
    cmap="RdYlGn",
    vmin=-0.3,
    vmax=0.15
).format("{:.3f}")

## Model Comparison

Across all models, several consistent patterns emerge:

1. **Nonlinear models outperform linear regression**
    - Random forest and gradient boosting substantially improve predictie performance

2. **Drought variables improve within-region performance**
    - Adding drought metrics consistently increases R2

3. **Transfer performance is poor across all models**
    - Models trained in the Corn Belt fail to generalize to the Great Plains
    - Reverse trnasfer performs slightly better but remains weak

4. **Transfer is asymmetric**
    - Great Plains to Corn Belt performs better than Corn Belt to Great Plains
    - This pattern is consistent across multiple model types

Findings suggest that climate yield relationships are region specific, not easily transferable even within the US

## Feature Importance

Permutation feature importance shows clear regional differences:

- In the **Corn Belt**, temperature and drought intensity are th emost important predictors
- In the **Great Plains**, temperature and precipitation dominate, with drought playing a smaller role

Under transfer:
- Feature importance weakens substantially
- Some variables, like drought frequency, can even negatively impact predictions

These results suggest that the role of climate and drought differ across regions despite them planting the same 

In [23]:
fi = pd.read_csv("../outputs/feature_importance_comparison.csv")
within_fi = fi[
    ((fi["trained_on"] == "corn_belt") & (fi["evaluated_on"] == "corn_belt")) |
    ((fi["trained_on"] == "great_plains") & (fi["evaluated_on"] == "great_plains"))
].copy()

within_fi["region"] = within_fi["trained_on"].map({
    "corn_belt": "Corn Belt",
    "great_plains": "Great Plains"
})

within_table = within_fi.pivot(
    index="feature",
    columns="region",
    values="importance_mean"
).round(3)

within_table

region,Corn Belt,Great Plains
feature,,
drought_freq_d2plus,0.057,0.061
drought_intensity_d2plus,0.247,0.150
prism_ppt_may_sep_total,0.140,0.553
prism_tmax_may_sep_mean,0.421,0.791


Within-region feature importance shows that temperature is the most important predictor in both regions. However, precipitation is much more important in the Great Plains, while drought intensity plays a larger role in the Corn Belt.

In [24]:
transfer_fi = fi[
    ((fi["trained_on"] == "corn_belt") & (fi["evaluated_on"] == "great_plains")) |
    ((fi["trained_on"] == "great_plains") & (fi["evaluated_on"] == "corn_belt"))
].copy()

transfer_fi["direction"] = transfer_fi.apply(
    lambda row: "CB → GP" if row["trained_on"] == "corn_belt" else "GP → CB",
    axis=1
)

transfer_table_fi = transfer_fi.pivot(
    index="feature",
    columns="direction",
    values="importance_mean"
).round(3)

transfer_table_fi

direction,CB → GP,GP → CB
feature,,
drought_freq_d2plus,-0.036,0.013
drought_intensity_d2plus,0.061,0.072
prism_ppt_may_sep_total,0.064,0.140
prism_tmax_may_sep_mean,0.167,0.352


Under transfer, feature importance patterns weaken and shift. The Corn Belt-trained model places negative importance on drought frequency when evaluated in the Great Plains, suggesting that this feature is actively misleading out-of-region predictions. In contrast, the Great Plains-trained model retains positive importance across all features when evaluated in the Corn Belt, which helps explain why reverse transfer performs somewhat better.

# still need to fix up

- more hyper-param tuning or justification for not doing more
- could try another type of model
- any params can think of and find to add? see if have more influence